# Reproduce the headline table and the heatmap

This notebook recomputes the study's results table (`tables.csv`) and its two figures from the **published per-item labels**. It runs on a CPU in a few minutes and needs no GPU, no model, no API key and no network.

You need the public export folder (`export/`, which holds `MANIFEST.json` and `dataset/`) and the small `msd` package from this repository (`src/msd`).

**Install.** From a checkout of the repository:

    pip install -e .

From a downloaded copy of the export and of `src/msd` only, no checkout:

    pip install numpy "matplotlib>=3.10"
    PYTHONPATH=src jupyter lab        # or just keep a src/ folder beside this notebook

**What a pass means.** The sha256 of the recomputed `tables.csv` must equal the one `MANIFEST.json` records for the published table. If it does not, the notebook stops with an error and names the rows and numbers that differ.

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("export")  # the folder export_public.py wrote, or a complete download of it (it holds MANIFEST.json)
OUT_DIR = Path("reproduced")  # the recomputed files go here, never into the export

if not EXPORT_DIR.is_dir() and (Path("..") / EXPORT_DIR).is_dir():  # Jupyter starts a notebook in its own folder, notebooks/
    EXPORT_DIR = Path("..") / EXPORT_DIR

The next cell imports `msd`. If it is not installed it uses the `src/` folder in the folder you started Jupyter from, or one level up.

In [ ]:
import sys

try:
    from msd import reproduce
except ModuleNotFoundError:  # a downloaded copy without `pip install -e .`
    src = next((p for p in (Path("src"), Path("../src")) if (p / "msd").is_dir()), None)
    assert src is not None, "msd is not installed and there is no src/msd here or one folder up: see the install lines at the top"
    sys.path.insert(0, str(src.resolve()))
    from msd import reproduce

## What is recomputed, and what is a recorded decision

The export holds one row per judged item (prompt id, language, arm, alpha, the judge's label and a short excerpt) and one row per task item (right or wrong). It does **not** hold the models' completions: they are not published.

**Recomputed here from those rows, never copied:** every rate (attack success, over-refusal, task accuracy), its bootstrap interval, the number of items, the count of items with no judged outcome, the `status` column, and the paired change per language printed beside the heatmap. The same code that built the original table does it (`msd.aggregate`, `msd.stats`, `msd.report`). The public rows do not say whether a prompt was harmful or benign; that is read back from the prompt files by prompt id.

**Recorded, read from `reliability.json` in the export:** decisions that rest on data that is not published, so they cannot be redone from the public rows.
- Which cells were left out of the table: the sanity gate reads completions, and the judge-error allowance sits in each cell's private `eval.json`.
- Which languages are marked low-confidence: that needs the second judge's labels.
- Which alpha was chosen on the dev split, and which two arms the heatmap compares.

They are checked where the public rows allow it (the judge, the alpha, the arms), and the notebook stops if one is missing or does not fit the rows. Whether a recorded decision was right is not something this notebook can show. It shows that the published table follows from the published rows and those decisions.

## 1. Is this the export that was published?

`MANIFEST.json` lists every file of the export with its sha256. Each file is hashed again and compared.

In [ ]:
problems = reproduce.verify_manifest(EXPORT_DIR)
if problems:
    print("\n".join(problems))
    raise AssertionError(
        f"{len(problems)} file(s) of the export are missing or were changed after it was built:\n" + "\n".join(problems)
        + "\nDownload the export again."
    )
print(f"every file of {EXPORT_DIR} is as exported (checked against MANIFEST.json)")

## 2. Recompute `tables.csv`

This reads the public rows of the report split (the table comes from it alone) and writes the table with the same writer the report used.

In [ ]:
public = reproduce.table_from_public(EXPORT_DIR)
recomputed = public.write_tables(OUT_DIR / "tables.csv")
print(f"read {public.n_verdict_rows} verdict rows and {public.n_task_rows} task rows (judge {public.judge_id})")
print(f"recomputed {len(public.rows)} table rows -> {recomputed}")

## 3. Does it match?

The recomputed file must hash to what `MANIFEST.json` records and to the exported file itself. If not, the cell prints where the two tables part and fails.

In [ ]:
exported = EXPORT_DIR / reproduce.TABLES_FILE
recomputed_sha = reproduce.sha256_file(recomputed)
manifest_sha = reproduce.manifest_tables_sha256(EXPORT_DIR)
exported_sha = reproduce.sha256_file(exported)
print("recomputed", recomputed_sha)
print("manifest  ", manifest_sha)
print("exported  ", exported_sha)
if recomputed_sha == manifest_sha == exported_sha:
    print("MATCH: the recomputed tables.csv is byte for byte the published one")
else:
    print("MISMATCH: the recomputed table is not the published one. Where they part:")
    print("\n".join(reproduce.explain_difference(recomputed, exported)
                    or ["the recomputed table equals the exported file, but MANIFEST.json records another sha256 for it"]))
    raise AssertionError("the recomputed tables.csv does not match the published one (the sha256 differs): the lines above say which numbers differ")

## 4. The heatmap

It compares attack success (lower is safer) for the two arms the report chose (a recorded decision), one row per language. The cell redraws it from the recomputed table and checks the **plotted numbers** (each cell's rate, interval and n) against the exported table. PNG bytes are deliberately not compared: matplotlib draws slightly differently across versions and platforms, so a correct recomputation can give different bytes. The paired change printed on the right of the heatmap comes from the verdict pairs; no exported file records it, so it is shown here but only the published picture can confirm it.

In [ ]:
from IPython.display import Image, display

figures = {path.name: path for path in public.write_figures(OUT_DIR)}
ours = reproduce.figure_numbers(reproduce.read_table_csv(recomputed), public.heat, public.chosen_alpha)
theirs = reproduce.figure_numbers(reproduce.read_table_csv(exported), public.heat, public.chosen_alpha)


def show(name):
    for title, path in (("recomputed", figures.get(name)), ("published", EXPORT_DIR / reproduce.AGGREGATES_ROOT / name)):
        if path is not None and path.is_file():
            print(title + ":")
            display(Image(filename=str(path)))


for lang, arm, alpha, rate, lo, hi, n in ours["heatmap"]:
    spread = "" if lo is None else f"  [{lo * 100:.1f}, {hi * 100:.1f}]"
    print(f"{lang:9}{arm:4}{'-' if alpha is None else alpha:>5}  attack success {rate * 100:5.1f}%{spread}  n={n}")
for lang, (change, lo, hi) in public.deltas.items():
    print(f"{lang:9}paired change {change * 100:+.1f} points  [{lo * 100:+.1f}, {hi * 100:+.1f}]")
assert ours["heatmap"] == theirs["heatmap"], "the plotted heatmap numbers differ from the exported table"
print("heatmap numbers match the exported table")
for note in public.notes:  # anything the report would have warned about while drawing
    print("NOTE:", note)
show("heatmap.png")

## 5. The alpha curve

Safety against task accuracy along the alpha grid of the repair: one line per language, one point per alpha, with the two unrepaired models as single markers. The ring marks the alpha chosen on the dev split (recorded). A language with no task items has no task accuracy, so it has no x and is not drawn; the figure says so. The numbers are compared the same way as the heatmap's.

In [ ]:
for lang, arm, alpha, accuracy, asr in ours["alpha_curve"]:
    print(f"{lang:9}{arm:4}{'-' if alpha is None else alpha:>5}  task accuracy {accuracy * 100:5.1f}%  attack success {asr * 100:5.1f}%")
assert ours["alpha_curve"] == theirs["alpha_curve"], "the plotted alpha-curve numbers differ from the exported table"
print("alpha-curve numbers match the exported table")
show("alpha_curve.png")

## Optional: regenerate and re-judge about 20 items on a free GPU (not guaranteed)

Everything above runs on a CPU and is expected to pass. This last section is different. It regenerates about 20 items with the published fine-tuned model, asks a judge to label them again, and counts how many labels equal the published ones. It is documented but **not guaranteed**. It needs:
- access to the gated Tiny Aya model: accept its licence on Hugging Face and set `HF_TOKEN` in the environment (a Kaggle or Colab secret works);
- a judge: `ANTHROPIC_API_KEY` in the environment for an `anthropic:` judge (it costs a small amount), or an open `hf:` judge on the GPU; the published labels were made by the judge `reliability.json` names;
- a checkout of this repository (it calls `scripts/run_eval.py`) and the id of the published model.

Free runtimes change, so it may need small fixes. The results will differ somewhat from run to run: a judge is not perfectly repeatable and the same model can write different text on another runtime. A few labels will differ and that is not a failure. Keys are read from the environment and never written into a cell. Leave `RUN_GPU = False` to skip it.

In [ ]:
RUN_GPU = False  # switch to True only on a GPU runtime with the access described above

if RUN_GPU:
    import os
    import subprocess

    ARM, LANG, N_ITEMS = "A1", "hi", 20
    MODEL = "YOUR-HF-USERNAME/published-A1-model"  # edit: the published fine-tuned model (a repo id or a local folder)
    JUDGE = "anthropic:claude-sonnet-5-5"  # edit: the judge named in reliability.json, or "hf:<model id>"
    script = next((p for p in (Path("scripts/run_eval.py"), Path("../scripts/run_eval.py")) if p.is_file()), None)
    assert script is not None, "scripts/run_eval.py not found: this section needs a checkout of the repository"
    assert "YOUR-HF-USERNAME" not in MODEL, "edit MODEL to the id of the published model"
    assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set: accept the Tiny Aya licence and add it as a secret"
    assert JUDGE.startswith("hf:") or os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set"
    print("the published labels were made by:", public.judge_id)
    work = OUT_DIR / "gpu"
    testset = reproduce.rebuild_testset(EXPORT_DIR, work / "testset", lang=LANG, limit=N_ITEMS)
    subprocess.run([
        sys.executable, str(script), "--arm", ARM, "--split", "report", "--model", MODEL, "--judge", JUDGE,
        "--langs", LANG, "--limit", str(N_ITEMS), "--stages", "generate,judge", "--testset-dir", str(testset),
        "--ft-dir", str(work / "no-task-items"), "--out", str(work / "fresh"),
    ], check=True)
    fresh = work / "fresh" / "report" / ARM / "none" / f"{LANG}.verdicts.jsonl"
    result = reproduce.compare_with_published(EXPORT_DIR, fresh, arm=ARM, alpha=None, lang=LANG)
    print(f"{result['same']} of {result['compared']} labels equal the published ones (some difference is expected)")
else:
    print("GPU section skipped (RUN_GPU = False)")